In [1]:
import pandas as pd

df = pd.read_csv('../data/netflix_titles.csv')
print(df.shape)
df.head()

(8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [2]:
# Har column mein kitni values missing hain
print("MISSING VALUES:")
print(df.isnull().sum())

print("\nDUPLICATE ROWS:", df.duplicated().sum())

print("\nCOLUMN TYPES:")
print(df.dtypes)


MISSING VALUES:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

DUPLICATE ROWS: 0

COLUMN TYPES:
show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object


In [3]:
# Text columns mein khali jagah "Unknown" se bharo
for col in ['director', 'cast', 'country', 'rating']:
    df[col] = df[col].fillna('Unknown')

print("Ab missing values:")
print(df.isnull().sum())

Ab missing values:
show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           0
duration         3
listed_in        0
description      0
dtype: int64


In [4]:
# date_added ko asli date banao
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')

# Date se saal aur mahina alag karo
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

print(df[['date_added', 'year_added', 'month_added']].head())
print("\nMissing date_added:", df['date_added'].isnull().sum())

  date_added  year_added  month_added
0 2021-09-25      2021.0          9.0
1 2021-09-24      2021.0          9.0
2 2021-09-24      2021.0          9.0
3 2021-09-24      2021.0          9.0
4 2021-09-24      2021.0          9.0

Missing date_added: 10


In [5]:
print(df['duration'].head(10))

0       90 min
1    2 Seasons
2     1 Season
3     1 Season
4    2 Seasons
5     1 Season
6       91 min
7      125 min
8    9 Seasons
9      104 min
Name: duration, dtype: str


In [6]:
# Number aur unit alag karo
df['duration_num'] = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_unit'] = df['duration'].str.extract(r'([a-zA-Z]+)')

# "Season" aur "Seasons" ko ek jaisa karo
df['duration_unit'] = df['duration_unit'].replace({'Seasons': 'Season'})

print(df[['type', 'duration', 'duration_num', 'duration_unit']].head(10))
print("\nUnits:")
print(df['duration_unit'].value_counts())

      type   duration  duration_num duration_unit
0    Movie     90 min          90.0           min
1  TV Show  2 Seasons           2.0        Season
2  TV Show   1 Season           1.0        Season
3  TV Show   1 Season           1.0        Season
4  TV Show  2 Seasons           2.0        Season
5  TV Show   1 Season           1.0        Season
6    Movie     91 min          91.0           min
7    Movie    125 min         125.0           min
8  TV Show  9 Seasons           9.0        Season
9    Movie    104 min         104.0           min

Units:
duration_unit
min       6128
Season    2676
Name: count, dtype: int64


In [7]:
# Kai countries hain to pehla wala alag nikalo
df['primary_country'] = df['country'].str.split(',').str[0].str.strip()

# Genres ko list banao
df['genres'] = df['listed_in'].str.split(', ')

print(df[['country', 'primary_country']].head())
print("\nTop 5 countries:")
print(df['primary_country'].value_counts().head())

         country primary_country
0  United States   United States
1   South Africa    South Africa
2        Unknown         Unknown
3        Unknown         Unknown
4          India           India

Top 5 countries:
primary_country
United States     3211
India             1008
Unknown            831
United Kingdom     628
Canada             271
Name: count, dtype: int64


In [8]:
df.to_csv('../data/netflix_clean.csv', index=False)
print("Save ho gaya!")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])

Save ho gaya!
Rows: 8807 | Columns: 18


# Task 1 — Data Cleaning & Preprocessing

**Dataset:** Netflix Movies and TV Shows (8807 rows, 12 columns)

## Masail jo mile
| Masla | Tafseel | Hal |
|---|---|---|
| Missing director | 2634 (30%) | "Unknown" — rows delete nahi kiye |
| Missing country | 831 | "Unknown" |
| Missing cast | 825 | "Unknown" |
| date_added text tha | spaces bhi the | strip + datetime |
| duration mein 2 units | "90 min" aur "2 Seasons" | number + unit alag kiye |
| country/genre multi-value | "US, India, France" | pehla country alag, genres list |

## Ahem faisla
Director wali 2634 rows delete karne se 30% data zaya hota, jabke un rows
mein title, country, genre sab mojood tha. Is liye "Unknown" bhara.

**Natija:** 8807 rows, 18 columns → `data/netflix_clean.csv`
